# 05 · Race, mental health, and other factors

Two follow-up questions:

1. **Who died?** The Medical Examiner records the race and ethnicity of each person. Comparing age-adjusted death rates across groups shows how unevenly the fentanyl era landed.
2. **What else goes with high overdose rates?** Beyond poverty and car access, this tests mental health, housing insecurity, unemployment, and education at the tract level.

Framing matters here. Differences by race describe how the crisis played out across communities with very different access to treatment, harm reduction, and resources. They say nothing about individuals or about any group's behavior.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import model, plots
from src.config import PROCESSED_DIR

plots.set_style()

## 1. Age-adjusted death rates by race and ethnicity

Rates are age-adjusted to the 2000 U.S. standard population, so groups with different age structures compare fairly, and each year uses that year's ACS population. The series starts in 2016 because the ME wasn't reliably recording Hispanic ethnicity before then (see `docs/methodology.md`, D7).

In [ ]:
rates = pd.read_csv(PROCESSED_DIR / "age_adjusted_rates_by_race.csv")
wide = rates.pivot(index="year", columns="race_group", values="age_adjusted_rate")
wide["black_to_white_ratio"] = wide["Black"] / wide["White (non-Hispanic)"]
wide.round(1)

In [ ]:
# Fixed color per group, in palette order, so identity never depends on rank
group_colors = {"Black": plots.BLUE, "White (non-Hispanic)": plots.ORANGE, "Hispanic (any race)": plots.AQUA}

fig, ax = plt.subplots(figsize=(9, 5))
for group, color in group_colors.items():
    series = rates[rates["race_group"] == group].sort_values("year")
    ax.fill_between(series["year"], series["ci_low"], series["ci_high"], color=color, alpha=0.15, linewidth=0)
    ax.plot(series["year"], series["age_adjusted_rate"], color=color, linewidth=2, marker="o", markersize=5,
            markeredgecolor=plots.SURFACE, label=group)
    last = series.iloc[-1]
    ax.text(last["year"] + 0.15, last["age_adjusted_rate"], group, va="center", fontsize=9, color=plots.TEXT_PRIMARY)

# Call out how the gap widened between the first year and the 2023 peak ratio
peak_year = wide["black_to_white_ratio"].idxmax()
for year in [wide.index.min(), peak_year]:
    ratio = wide.loc[year, "black_to_white_ratio"]
    ax.annotate(f"{ratio:.1f}x", xy=(year, wide.loc[year, "Black"]), xytext=(0, 10),
                textcoords="offset points", ha="center", fontsize=9, color=plots.TEXT_SECONDARY)

ax.set_title("The fentanyl era widened the racial gap in overdose deaths")
ax.set_ylabel("Deaths per 100k, age-adjusted")
ax.set_xticks(wide.index)
ax.set_xlim(wide.index.min() - 0.4, wide.index.max() + 2.2)
ax.legend(loc="upper left")
plots.add_source_note(fig, "Black to White rate ratio labeled above the Black line. Shaded bands are 95% CIs. "
                           "Sources: Cook County ME; ACS 1-year estimates. Age-adjusted to the 2000 U.S. standard.")
plots.save_figure(fig, "rates_by_race.png")
plt.show()

## 2. Which neighborhood factors go with high overdose rates?

A first screen: the rank correlation (Spearman) between each factor and the tract overdose rate. This looks at one factor at a time, so factors that travel together will all look strong.

Mental health and several social needs come from **CDC PLACES**, which estimates them for every tract with a model. That model uses each tract's demographics, including poverty and race, so PLACES measures partly repeat what the Census variables already say.

In [ ]:
tract_table = pd.read_csv(PROCESSED_DIR / "tract_table.csv", dtype={"GEOID": str})
tracts = tract_table[tract_table["population_2020"] >= 500]

factor_labels = {
    "pct_lack_transportation": "Transportation barriers (PLACES)",
    "pct_housing_insecurity": "Housing insecurity (PLACES)",
    "pct_frequent_mental_distress": "Frequent mental distress (PLACES)",
    "pct_loneliness": "Loneliness (PLACES)",
    "pct_smoking": "Smoking (PLACES)",
    "pct_disability": "Disability (PLACES)",
    "pct_poverty": "Poverty (ACS)",
    "pct_nh_black": "Black residents (ACS)",
    "pct_no_vehicle": "No vehicle (ACS)",
    "pct_unemployed": "Unemployment (ACS)",
    "pct_no_high_school": "No high school diploma (ACS)",
    "pct_uninsured": "Uninsured (ACS)",
    "pct_depression": "Diagnosed depression (PLACES)",
    "pct_hispanic": "Hispanic residents (ACS)",
    "pct_binge_drinking": "Binge drinking (PLACES)",
    "median_household_income": "Median household income (ACS)",
}
correlations = (
    tracts[list(factor_labels)]
    .corrwith(tracts["overdose_rate_per_100k"], method="spearman")
    .rename(index=factor_labels)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(correlations.index, correlations.values, color=plots.BLUE, height=0.65)
for y, value in enumerate(correlations.values):
    offset = 0.02 if value >= 0 else -0.02
    ax.text(value + offset, y, f"{value:.2f}", va="center", ha="left" if value >= 0 else "right", fontsize=8)
ax.axvline(0, color=plots.TEXT_SECONDARY, linewidth=1)
ax.set_xlim(-0.85, 0.85)
ax.grid(axis="x")
ax.grid(axis="y", visible=False)
ax.set_xlabel("Rank correlation with tract overdose death rate")
ax.set_title("Hardship measures move with overdose rates; diagnosed depression doesn't")
plots.add_source_note(fig, f"Spearman correlations across {len(tracts):,} tracts with 500+ residents. "
                           "Sources: Cook County ME, ACS 2019-2023, CDC PLACES 2025.")
plots.save_figure(fig, "factor_correlations.png")
plt.show()

How much do the PLACES measures overlap with poverty and race? High overlap means that when both go in a model, the data can't tell them apart.

In [ ]:
places_measures = ["pct_frequent_mental_distress", "pct_housing_insecurity", "pct_smoking",
                   "pct_depression", "pct_binge_drinking"]
pd.DataFrame({
    "correlation with poverty": tracts[places_measures].corrwith(tracts["pct_poverty"], method="spearman"),
    "correlation with % Black": tracts[places_measures].corrwith(tracts["pct_nh_black"], method="spearman"),
}).rename(index=factor_labels).round(2)

## 3. Do they add anything once everything is in the model together?

Start from the Phase 4 negative binomial model (poverty, no vehicle, uninsured, race and ethnicity, treatment sites per resident). Add each new factor one at a time and check three things:

- does model fit improve (lower AIC)?
- how strong is the new factor, holding the others fixed?
- what happens to poverty and car access when it goes in?

To compare factors on equal terms, each added factor's rate ratio is **per standard deviation**, not per 10 points. Mental distress only ranges from about 10% to 27% across tracts, so a 10-point step would be misleadingly large.

In [ ]:
added_factors = {
    "pct_frequent_mental_distress": "Frequent mental distress",
    "pct_depression": "Diagnosed depression",
    "pct_housing_insecurity": "Housing insecurity",
    "pct_unemployed": "Unemployment",
    "pct_no_high_school": "No high school diploma",
}
data = model.prepare_model_data(tract_table).dropna(subset=list(added_factors)).reset_index(drop=True)
for column in added_factors:
    # z-score: 0 = county average tract, 1 = one standard deviation above it
    data[f"{column}_sd"] = (data[column] - data[column].mean()) / data[column].std()

base_result, _ = model.fit_negative_binomial(data)
rows = [{"model": "Base model (Phase 4)", "AIC": base_result.aic,
         "added factor rate ratio (per SD)": None,
         "poverty (per 10 pts)": np.exp(base_result.params["pct_poverty_per10"]),
         "no vehicle (per 10 pts)": np.exp(base_result.params["pct_no_vehicle_per10"])}]

for column, label in added_factors.items():
    term = f"{column}_sd"
    result, _ = model.fit_negative_binomial(data, model.model_formula() + f" + {term}")
    low, high = np.exp(result.conf_int().loc[term])
    rows.append({
        "model": f"+ {label}",
        "AIC": result.aic,
        "added factor rate ratio (per SD)": f"{np.exp(result.params[term]):.2f} ({low:.2f} to {high:.2f}), p = {result.pvalues[term]:.3f}",
        "poverty (per 10 pts)": np.exp(result.params["pct_poverty_per10"]),
        "no vehicle (per 10 pts)": np.exp(result.params["pct_no_vehicle_per10"]),
    })

comparison = pd.DataFrame(rows).set_index("model")
comparison["AIC"] = comparison["AIC"].round(0)
comparison.round(2)

In [ ]:
# Standard deviations, to translate "per SD" back into percentage points
data[list(added_factors)].std().rename(index=added_factors).round(1).rename("1 SD in percentage points").to_frame()

## What this shows

- **The fentanyl era hit Black residents far harder.** In 2016, Black Cook County residents died of overdoses at 1.5 times the White rate (age-adjusted). By 2023 it was 3.6 times: 84.9 vs 23.4 deaths per 100k. The White rate barely moved through the fentanyl wave (21.6 in 2016, 28.8 at its 2022 peak), while the Black rate nearly tripled (32.7 to 88.9). Rates have fallen for every group since, but the Black rate is still 2.6 times the White rate in 2025.
- **Mental distress is one of the strongest correlates, and it's tangled up with poverty.** Tracts where more adults report frequent mental distress have much higher overdose rates (1.45 times per standard deviation, holding the other factors fixed). But when it enters the model, poverty's effect drops to nothing. Housing insecurity does the same thing. Poverty, housing insecurity, and mental distress rise and fall together so closely across Cook County tracts (correlations of about 0.77) that this data can't separate them. The fair reading is that they form one cluster of hardship tied to overdose deaths, not three separate causes.
- **Diagnosed depression tells a different story from distress.** On its own it has no relationship with overdose rates, even though distress does. Being diagnosed takes a visit to a clinician, so diagnosis rates partly track access to care. Once poverty and race are held fixed, diagnosed depression turns positive. That fits underdiagnosis in poorer tracts hiding the real pattern, though this data can't prove it.
- **Car access holds up no matter what else is in the model.** Every version gives about the same rate ratio for households without a vehicle (1.19 to 1.21 per 10 points). It's the most stable finding in the project and ties back to how hard it is to reach daily treatment without a car.
- **Unemployment and education add little** once poverty and the other factors are in the model.

**Caveats:** PLACES values are model-based estimates that partly use tract demographics as inputs, which is part of why they overlap with poverty. The ME records don't capture mental illness for individual decedents (one mention of schizophrenia across 17,000 deaths), so mental health can only be looked at by neighborhood here, not person by person.